In [ ]:
from hpo_rl.experiments.run_experiment import run_n_experiments
from hpo_rl.models.simple_cnn import SimpleCNN
from hpo_rl.trainers.torch_trainer import TorchTrainer
from hpo_rl.data_processing.processors import pytorch_mnist_processor
from hpo_rl.nets.masked_net import MaskedNet
from hpo_rl.nets.base_net import BaseNet
from hpo_rl.nets.masked_actor import MaskedDiscreteActor
from hpo_rl.nets.recurrent_net import RecurrentBaseNet
from hpo_rl.nets.recurrent_actor import MaskedRecurrentDiscreteActor
from hpo_rl.nets.recurrent_critic import RecurrentCritic
from hpo_rl.nets.masked_recurrent_net import MaskedRecurrentNet
from torch.optim import Adam
from tianshou.algorithm.modelfree.reinforce import ProbabilisticActorPolicy
from tianshou.algorithm.modelfree.dqn import DiscreteQLearningPolicy
from tianshou.algorithm.modelfree.c51 import C51Policy
from tianshou.utils.net.discrete import DiscreteActor
from tianshou.utils.net.discrete import DiscreteCritic
from tianshou.utils.net.continuous import ContinuousActorProbabilistic
from tianshou.utils.net.continuous import ContinuousCritic
import torch
from tianshou.utils.net.common import Net
from tianshou.utils.net.common import Recurrent
from tianshou.algorithm.modelfree.sac import SACPolicy
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm

c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=100, n_params=128):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, n_params) 
        self.fc2 = nn.Linear(n_params, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) 
        x = self.pool(F.relu(self.conv2(x))) 
        x = x.view(-1, 64 * 8 * 8) 
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def objective_function(config, dict_config):
    param_values = {}
    for name in dict_config.keys():
        param_values[name] = config[name]

    n_params = param_values["n_params"]
    lr = param_values["lr"]
    batch_size = int(param_values["batch_size"])
    optimizer_name = param_values["optimizer"]

    transform = transforms.ToTensor()
    
    try:
        dataset = datasets.CIFAR100(root='./tmp_data', train=True, download=True, transform=transform)
    except:
        dataset = datasets.CIFAR100(root='./tmp_data', train=True, download=False, transform=transform)
        
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = SimpleCNN(num_classes=100, n_params=n_params) 
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr)

    model.train()
    
    sub_bar = tqdm(total=int(2),desc="Model training", position=1, leave=False)
    
    for epoch in range(int(2)):
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
        sub_bar.update(1)
    
    sub_bar.close()

    model.eval()
    val_loss, correct = 0.0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            val_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / len(val_dataset)

    print(f"Config: {param_values}, ValLoss: {avg_val_loss:.4f}, ValAcc: {val_accuracy:.4f}")

    return avg_val_loss

In [9]:
config_recurrent_ppo = {
    "full_args": {
            "algorithm":
            {
                "name": "recurrent_ppo",
                "gamma": 0.97,                # shorter horizon: 1/(1-0.97)≈33 steps — достаточно для HPO
                "gae_lambda": 0.95, 
                "seq_len": 10,                # MUST divide max_steps (200 % 10 = 0)
                "vf_coef": 0.5,               # стандартное значение: critic важен для качественных advantages
                "ent_coef": 0.01,             # exploration: не слишком много, чтобы не мешать сходимости
                "max_grad_norm": 0.5,         # gradient clipping — КРИТИЧНО для RNN!
                "value_clip": True,           # стабилизация value function
                "return_scaling": True,       # нормализация returns по running std — критик работает с любым масштабом
                "recompute_advantage": True,  # пересчёт advantages после каждого update — точнее для RNN
            },  
            "optim":
            {
                "name": "TorchOptimizerFactory",
                "optim_class": torch.optim.Adam,
                "lr": 3e-4,  
            },
            "net":
            {
                "actor": MaskedRecurrentDiscreteActor,
                "critic": RecurrentCritic, 
                "net": RecurrentBaseNet,
                "hidden_layer_size": 64,      # 64 вместо 128: obs_dim=5, 12.8x ratio — лучше для маленьких задач
            },
            "trainer":
            {
                "max_epochs": 50,            # больше эпох для delta rewards (меньший сигнал)
                "epoch_num_steps": 4000,       # кратно collection (4000/2000=2 collects)
                "batch_size": 20,             # chunks: 2000/10=200 chunks → 10 minibatch
                "collection_step_num_env_steps": 2000,  # 10 полных эпизодов → больше данных для GAE
                "update_step_num_repetitions": 8, # 8 прохождений по данным (было 4) — больше обновлений
            },
            "policy":
            {
                "class": ProbabilisticActorPolicy,
                "dist_fn": lambda x: torch.distributions.Categorical(logits=x),
                "action_scaling": False,
            },
            "inference": 
            {
                "n_episode": 1,
                "reset_before_collect": True,
            },
            "num_training_envs": 20, 
            "num_test_envs": 20,
            # "load_checkpoint": "log/recurrent_ppo/20260226-201335/best_policy.pth",

        },
        "env": {
            "name": "new_cycle_move_pipeline",
            "num_bins": 500,
            "max_steps": 200,
            "step_sizes": [1, 2, 5, 10, 25, 50],
            "history_window": 0,
            "reward_mode": "absolute"          
        },
        "backend": {"name": "function", "function": "rastrigin", "dimensions": 2},
        # {
        #     "name": "sequential",
        #     "mode": "random",  # по умолчанию
        #     "backends": [
        #         {"name": "function", "function": "rastrigin", "dimensions": 2},
        #         {"name": "function", "function": "rosenbrock", "dimensions": 2},
        #         {"name": "function", "function": "schwefel", "dimensions": 2},
        #         # {"name": "function", "function": "goldstein_price", "dimensions": 2},
        #     ]
        # }
    }

In [10]:
run_n_experiments(config_recurrent_ppo, 3, inference_only=False)

wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


rastrigin: dims=2, bounds=(-5.12, 5.12), opt=0.000000


wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Model saved locally to: log/recurrent_ppo/20260227-230858\best_policy.pth
Initial test step: test_reward: -714.870754 ± 36.431553, best_reward: -714.870754 ± 36.431553 in #0


Epoch #1: 100%|##########| 4000/4000 [00:02<00:00, 1665.90it/s, env_episode=20, env_step=4000, len=100, n_ep=20, n_st=2000, rew=-723.56, update_step=2]


...

Saved: logs\recurrent_ppo\20260227_230858\3d_0.png, logs\recurrent_ppo\20260227_230858\3d_0.pgf
Saved: logs\recurrent_ppo\20260227_230858\trajectory_0.png, logs\recurrent_ppo\20260227_230858\trajectory_0.pgf
Saved TEX history: logs\recurrent_ppo\20260227_230858\history_table_0.tex
Saved CSV history: logs\recurrent_ppo\20260227_230858\history_0.csv


c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tianshou\data\collector.py:351: UserWarning: Single environment detected, wrap to DummyVectorEnv.
  warnings.warn("Single environment detected, wrap to DummyVectorEnv.")


Saved: logs\recurrent_ppo\20260227_230858\3d_1.png, logs\recurrent_ppo\20260227_230858\3d_1.pgf
Saved: logs\recurrent_ppo\20260227_230858\trajectory_1.png, logs\recurrent_ppo\20260227_230858\trajectory_1.pgf
Saved TEX history: logs\recurrent_ppo\20260227_230858\history_table_1.tex
Saved CSV history: logs\recurrent_ppo\20260227_230858\history_1.csv


c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tianshou\data\collector.py:351: UserWarning: Single environment detected, wrap to DummyVectorEnv.
  warnings.warn("Single environment detected, wrap to DummyVectorEnv.")


Saved: logs\recurrent_ppo\20260227_230858\3d_2.png, logs\recurrent_ppo\20260227_230858\3d_2.pgf
Saved: logs\recurrent_ppo\20260227_230858\trajectory_2.png, logs\recurrent_ppo\20260227_230858\trajectory_2.pgf
Saved TEX history: logs\recurrent_ppo\20260227_230858\history_table_2.tex
Saved CSV history: logs\recurrent_ppo\20260227_230858\history_2.csv
Saved median/best/worst: logs\recurrent_ppo\20260227_230858\inference_results.json


In [3]:
config_recurrent_dqn = {
    "full_args": {
        "algorithm":
        {
            "name": "recurrent_dqn",
            "gamma": 0.99,
            # "seq_len": 10,
            "target_update_freq": 320,
            # "n_step_return_horizon": 3,
        },
        "buffer":
        {
            "total_size": 20000,
            "buffer_num": 1,
            "stack_num": 1
        },  
        "optim":
        {
            "name": "TorchOptimizerFactory",
            "optim_class": Adam,
            "lr": 1e-3,
        },
        "net":
        {
            # "actor": DiscreteActor,
            # "critic": DiscreteCritic, 
            "hidden_sizes": [64, 64],
            "net": MaskedRecurrentNet,
            "rnn_layers": 1
        },
        "trainer":
        {
            "max_epochs": 50,
            "epoch_num_steps": 4000,
            "batch_size": 20,
            "collection_step_num_env_steps": 200,
            # "update_step_num_repetitions": 5,
            # "test_in_training": True,
            # "stop_fn": stop_fn
        },
        "policy":
        {
            "class": DiscreteQLearningPolicy,
            "eps_training": 0.1,
            "eps_inference": 0.0
        },
        "inference": 
        {
            "n_episode": 1,
            "reset_before_collect": True,
        },
        "num_training_envs": 20,
        "num_test_envs": 20,
    },
    "env": {
        "name": "new_cycle_move_pipeline",
        "num_bins": 500,
        "max_steps": 200,
        "step_sizes": [1, 2, 5, 10, 25, 50],
        "history_window": 3,
        "reward_mode": "absolute"
    },
    "backend": 
        {
            "name": "sequential",
            "mode": "shuffle",  # по умолчанию
            "backends": [
                {"name": "function", "function": "rastrigin", "dimensions": 2},
                {"name": "function", "function": "rosenbrock", "dimensions": 2},
                {"name": "function", "function": "schwefel", "dimensions": 2},
                # {"name": "function", "function": "ackley", "dimensions": 2},
            ]
        }
    }

In [ ]:
run_n_experiments(config_recurrent_dqn, 3, inference_only=False)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Administrator\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\wandb\analytics\sentry.py:268: DeprecationWarning: Read the `app_url` setting from the appropriate Settings object.
  app_url = wandb.util.app_url(tags["base_url"])  # type: ignore[index]
c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\wandb\analytics\sentry.py:268: DeprecationWarning: Read the `app_url` setting from the appropriate Settings object.
  app_url = wandb.util.app_url(tags["base_url"])  # type: ignore[index]
c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\wandb\analytics\sentry.py:268: DeprecationWarning: Read the `app_url` setting from the appropriate Settings object.
  ap

rastrigin: dims=2, bounds=(-5.12, 5.12), opt=0.000000
rosenbrock: dims=2, bounds=(-1.0, 1.0), opt=0.000000
schwefel: dims=2, bounds=(-500.0, 500.0), opt=0.000000
[SequentialBackend] New shuffle order: rastrigin, rosenbrock, schwefel
SequentialBackend: 3 backends (rastrigin, rosenbrock, schwefel), mode=shuffle, switch every epoch


wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Model saved locally to: log/recurrent_dqn/20260228-002230\best_policy.pth
Initial test step: test_reward: -671.288110 ± 87.374524, best_reward: -671.288110 ± 87.374524 in #0


Epoch #1:   0%|          | 0/4000 [00:00<?, ?it/s]

...

Epoch #24: 100%|██████████| 4000/4000 [00:28<00:00, 142.73it/s, env_episode=480, env_step=96000, len=200, n_ep=20, n_st=200, rew=-1319.72, update_step=480]



Epoch #24: test_reward: -1380.108529 ± 26.658451, best_reward: -153.899573 ± 40.365725 in #23


Epoch #25:   0%|          | 0/4000 [00:00<?, ?it/s]

[SequentialBackend] Epoch 25: switched to 'rastrigin'


Epoch #25: 100%|██████████| 4000/4000 [00:30<00:00, 130.37it/s, env_episode=500, env_step=100000, len=200, n_ep=20, n_st=200, rew=-320.65, update_step=500]



In [30]:
config_recurrent_dqn["full_args"]["load_checkpoint"] = "log/recurrent_dqn/20260227-234144\final_policy.pth"

In [31]:
run_n_experiments(config_recurrent_dqn, 3, inference_only=True)

wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`
c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tianshou\data\collector.py:351: UserWarning: Single environment detected, wrap to DummyVectorEnv.
  warnings.warn("Single environment detected, wrap to DummyVectorEnv.")
c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tianshou\data\collector.py:351: UserWarning: Single environment detected, wrap to DummyVectorEnv.
  warnings.warn("Single environment detected, wrap to DummyVectorEnv.")


ackley: dims=2, bounds=(-32.768, 32.768), opt=0.000000
SequentialBackend: 1 backends (ackley), mode=random, switch every epoch
[SequentialBackend] Manually switched to 'ackley' (idx=0)
[SequentialBackend] Manually switched to 'ackley' (idx=0)
Saved: logs\recurrent_dqn\20260228_001458\3d_0_0_ackley.png, logs\recurrent_dqn\20260228_001458\3d_0_0_ackley.pgf
Saved: logs\recurrent_dqn\20260228_001458\3d_0_0_ackley.png, logs\recurrent_dqn\20260228_001458\3d_0_0_ackley.pgf
Saved: logs\recurrent_dqn\20260228_001458\trajectory_0_0_ackley.png, logs\recurrent_dqn\20260228_001458\trajectory_0_0_ackley.pgf
Saved TEX history: logs\recurrent_dqn\20260228_001458\history_table_0_0_ackley.tex
Saved CSV history: logs\recurrent_dqn\20260228_001458\history_0_0_ackley.csv
Saved: logs\recurrent_dqn\20260228_001458\trajectory_0_0_ackley.png, logs\recurrent_dqn\20260228_001458\trajectory_0_0_ackley.pgf
Saved TEX history: logs\recurrent_dqn\20260228_001458\history_table_0_0_ackley.tex
Saved CSV history: logs\re

c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tianshou\data\collector.py:351: UserWarning: Single environment detected, wrap to DummyVectorEnv.
  warnings.warn("Single environment detected, wrap to DummyVectorEnv.")


[SequentialBackend] Manually switched to 'ackley' (idx=0)
Saved: logs\recurrent_dqn\20260228_001458\3d_1_0_ackley.png, logs\recurrent_dqn\20260228_001458\3d_1_0_ackley.pgf
Saved: logs\recurrent_dqn\20260228_001458\3d_1_0_ackley.png, logs\recurrent_dqn\20260228_001458\3d_1_0_ackley.pgf
Saved: logs\recurrent_dqn\20260228_001458\trajectory_1_0_ackley.png, logs\recurrent_dqn\20260228_001458\trajectory_1_0_ackley.pgf
Saved TEX history: logs\recurrent_dqn\20260228_001458\history_table_1_0_ackley.tex
Saved CSV history: logs\recurrent_dqn\20260228_001458\history_1_0_ackley.csv
Saved: logs\recurrent_dqn\20260228_001458\trajectory_1_0_ackley.png, logs\recurrent_dqn\20260228_001458\trajectory_1_0_ackley.pgf
Saved TEX history: logs\recurrent_dqn\20260228_001458\history_table_1_0_ackley.tex
Saved CSV history: logs\recurrent_dqn\20260228_001458\history_1_0_ackley.csv
Saved: logs\recurrent_dqn\20260228_001458\trajectory_1.png, logs\recurrent_dqn\20260228_001458\trajectory_1.pgf
Saved TEX history: log

c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tianshou\data\collector.py:351: UserWarning: Single environment detected, wrap to DummyVectorEnv.
  warnings.warn("Single environment detected, wrap to DummyVectorEnv.")


[SequentialBackend] Manually switched to 'ackley' (idx=0)
Saved: logs\recurrent_dqn\20260228_001458\3d_2_0_ackley.png, logs\recurrent_dqn\20260228_001458\3d_2_0_ackley.pgf
Saved: logs\recurrent_dqn\20260228_001458\3d_2_0_ackley.png, logs\recurrent_dqn\20260228_001458\3d_2_0_ackley.pgf
Saved: logs\recurrent_dqn\20260228_001458\trajectory_2_0_ackley.png, logs\recurrent_dqn\20260228_001458\trajectory_2_0_ackley.pgf
Saved TEX history: logs\recurrent_dqn\20260228_001458\history_table_2_0_ackley.tex
Saved CSV history: logs\recurrent_dqn\20260228_001458\history_2_0_ackley.csv
Saved: logs\recurrent_dqn\20260228_001458\trajectory_2_0_ackley.png, logs\recurrent_dqn\20260228_001458\trajectory_2_0_ackley.pgf
Saved TEX history: logs\recurrent_dqn\20260228_001458\history_table_2_0_ackley.tex
Saved CSV history: logs\recurrent_dqn\20260228_001458\history_2_0_ackley.csv
Saved: logs\recurrent_dqn\20260228_001458\trajectory_2.png, logs\recurrent_dqn\20260228_001458\trajectory_2.pgf
Saved TEX history: log

In [13]:
config_dqn = {
    "full_args": {
        "algorithm":
        {
            "name": "dqn",
            "gamma": 0.99,
            # "seq_len": 10,
            "target_update_freq": 320,
            # "n_step_return_horizon": 3,
        },
        "buffer":
        {
            "total_size": 20000,
            "buffer_num": 1,
            "stack_num": 1
        },  
        "optim":
        {
            "name": "TorchOptimizerFactory",
            "optim_class": Adam,
            "lr": 1e-3,
        },
        "net":
        {
            # "actor": DiscreteActor,
            # "critic": DiscreteCritic, 
            # "hidden_sizes": [64, 64],
            "net": MaskedNet,
            # "rnn_layers": 1
        },
        "trainer":
        {
            "max_epochs": 50,
            "epoch_num_steps": 4000,
            "batch_size": 20,
            "collection_step_num_env_steps": 200,
            # "update_step_num_repetitions": 5,
            # "test_in_training": True,
            # "stop_fn": stop_fn
        },
        "policy":
        {
            "class": DiscreteQLearningPolicy,
            "eps_training": 0.1,
            "eps_inference": 0.0
        },
        "inference": 
        {
            "n_episode": 1,
            "reset_before_collect": True,
        },
        "num_training_envs": 20,
        "num_test_envs": 20,
    },
    "env": {
        "name": "new_cycle_move_pipeline",
        "num_bins": 500,
        "max_steps": 200,
        "step_sizes": [1, 2, 5, 10, 25, 50],
        "history_window": 3,
        "reward_mode": "absolute"
    },
    "backend": {"name": "function", "function": "rastrigin", "dimensions": 2},
        # {
        #     "name": "sequential",
        #     "mode": "random",  # по умолчанию
        #     "backends": [
        #         {"name": "function", "function": "rastrigin", "dimensions": 2},
        #         {"name": "function", "function": "rosenbrock", "dimensions": 2},
        #         {"name": "function", "function": "schwefel", "dimensions": 2},
        #         # {"name": "function", "function": "goldstein_price", "dimensions": 2},
        #     ]
        # }
    }

In [14]:
run_n_experiments(config_dqn, 3, inference_only=False)


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


rastrigin: dims=2, bounds=(-5.12, 5.12), opt=0.000000


wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Model saved locally to: log/dqn/20260227-233843\best_policy.pth
Initial test step: test_reward: -804.198786 ± 7.204662, best_reward: -804.198786 ± 7.204662 in #0


Epoch #1:  80%|########  | 3200/4000 [00:10<00:02, 292.70it/s, env_episode=0, env_step=3200, n_ep=0, n_st=200, update_step=16]


KeyboardInterrupt: 

In [ ]:
config_ppo = {
    "full_args": {
            "algorithm":
            {
                "name": "ppo",
                "gamma": 0.97,                # shorter horizon: 1/(1-0.97)≈33 steps — достаточно для HPO
                "gae_lambda": 0.95, 
                # "seq_len": 10,                # MUST divide max_steps (200 % 10 = 0)
                "vf_coef": 0.5,               # стандартное значение: critic важен для качественных advantages
                "ent_coef": 0.01,             # exploration: не слишком много, чтобы не мешать сходимости
                "max_grad_norm": 0.5,         # gradient clipping — КРИТИЧНО для RNN!
                "value_clip": True,           # стабилизация value function
                "return_scaling": True,       # нормализация returns по running std — критик работает с любым масштабом
                "recompute_advantage": True,  # пересчёт advantages после каждого update — точнее для RNN
            },  
            "optim":
            {
                "name": "TorchOptimizerFactory",
                "optim_class": torch.optim.Adam,
                "lr": 3e-4,  
            },
            "net":
            {
                "actor": MaskedDiscreteActor,
                "critic": DiscreteCritic, 
                "net": BaseNet,
                "hidden_sizes": [256, 256, 256]
                # "hidden_layer_size": 64,      # 64 вместо 128: obs_dim=5, 12.8x ratio — лучше для маленьких задач
            },
            "trainer":
            {
                "max_epochs": 50,            # больше эпох для delta rewards (меньший сигнал)
                "epoch_num_steps": 4000,       # кратно collection (4000/2000=2 collects)
                "batch_size": 20,             # chunks: 2000/10=200 chunks → 10 minibatch
                "collection_step_num_env_steps": 2000,  # 10 полных эпизодов → больше данных для GAE
                "update_step_num_repetitions": 8, # 8 прохождений по данным (было 4) — больше обновлений
                "test_step_num_episodes": 20
            },
            "policy":
            {
                "class": ProbabilisticActorPolicy,
                "dist_fn": lambda x: torch.distributions.Categorical(logits=x),
                "action_scaling": False,
            },
            "inference": 
            {
                "n_episode": 1,
                "reset_before_collect": True,
            },
            "num_training_envs": 20, 
            "num_test_envs": 20,
            "load_checkpoint": "log/ppo/20260227-162201/final_policy.pth",

        },
        "env": {
            "name": "new_cycle_move_pipeline",
            "num_bins": 500,
            "max_steps": 200,
            "step_sizes": [1, 2, 5, 10, 25, 50],
            "history_window": 3,
            "reward_mode": "absolute",
            "obs_mode": "ohe"     
        },
        "backend": {"name": "function", "function": "rastrigin", "dimensions": 2},
        # {
        #     "name": "sequential",
        #     "mode": "random",  # по умолчанию
        #     "backends": [
        #         {"name": "function", "function": "rastrigin", "dimensions": 2},
        #         {"name": "function", "function": "rosenbrock", "dimensions": 2},
        #         {"name": "function", "function": "schwefel", "dimensions": 2},
        #         # {"name": "function", "function": "goldstein_price", "dimensions": 2},
        #     ]
        # }
    }


In [ ]:
run_n_experiments(config_ppo, 3, inference_only=False)
